# 09 - Trend & Seasonality Feature Engineering

## Objective

Marketing Mix Models must distinguish **marketing effects** from **natural business patterns**.

In this notebook we engineer temporal features that explain:

- Long-term trend
- Weekly seasonality
- Monthly seasonality
- Quarterly behaviour
- Rolling statistics
- Lag effects
- Fourier seasonality

These features help prevent the model from incorrectly attributing natural demand to marketing.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT=Path.cwd()
DATA=ROOT/"data"/"processed"/"marketing_mix_hill.csv"

df=pd.read_csv(DATA,parse_dates=["Week"])
df=df.sort_values("Week").reset_index(drop=True)

df.head()


## 1. Calendar Features

In [ ]:

iso=df["Week"].dt.isocalendar()

df["Year"]=df["Week"].dt.year
df["Month"]=df["Week"].dt.month
df["Quarter"]=df["Week"].dt.quarter
df["WeekOfYear"]=iso.week.astype(int)
df["DayOfYear"]=df["Week"].dt.dayofyear

display(df[["Week","Year","Month","Quarter","WeekOfYear"]].head())


## 2. Trend Index

In [ ]:

df["Trend"]=np.arange(len(df))
plt.figure(figsize=(12,3))
plt.plot(df["Week"],df["Trend"])
plt.title("Trend Index")
plt.grid(True)
plt.show()


## 3. Rolling Features

In [ ]:

df["Sales_MA_4"]=df["Sales"].rolling(4,min_periods=1).mean()
df["Sales_MA_12"]=df["Sales"].rolling(12,min_periods=1).mean()
df["Sales_STD_12"]=df["Sales"].rolling(12,min_periods=2).std()

plt.figure(figsize=(14,4))
plt.plot(df["Week"],df["Sales"],label="Sales",alpha=.5)
plt.plot(df["Week"],df["Sales_MA_12"],label="12 Week MA",linewidth=2)
plt.legend()
plt.show()


## 4. Lag Features

In [ ]:

for lag in [1,2,4,8]:
    df[f"Sales_Lag_{lag}"]=df["Sales"].shift(lag)

display(df.filter(regex="Sales_Lag").head(10))


## 5. Fourier Seasonality

In [ ]:

period=52
for k in range(1,4):
    df[f"sin_{k}"]=np.sin(2*np.pi*k*df["Trend"]/period)
    df[f"cos_{k}"]=np.cos(2*np.pi*k*df["Trend"]/period)

plt.figure(figsize=(12,3))
plt.plot(df["sin_1"],label="sin_1")
plt.plot(df["cos_1"],label="cos_1")
plt.legend()
plt.title("Fourier Terms")
plt.show()


## 6. Seasonal Aggregations

In [ ]:

monthly=df.groupby("Month")["Sales"].mean()
quarterly=df.groupby("Quarter")["Sales"].mean()

display(monthly.to_frame("Average Sales"))
display(quarterly.to_frame("Average Sales"))


## 7. Feature Correlation

In [ ]:

features=[
"Trend","Sales_MA_4","Sales_MA_12",
"Sales_Lag_1","Sales_Lag_4",
"sin_1","cos_1"
]

corr=df[features+["Sales"]].corr()

plt.figure(figsize=(8,6))
plt.imshow(corr,aspect="auto")
plt.xticks(range(len(corr.columns)),corr.columns,rotation=90)
plt.yticks(range(len(corr.columns)),corr.columns)
plt.colorbar()
plt.title("Feature Correlation")
plt.tight_layout()
plt.show()


## 8. Save Final Feature Dataset

In [ ]:

df=df.fillna(method="bfill").fillna(method="ffill")

OUT=ROOT/"data"/"processed"/"marketing_mix_features.csv"
df.to_csv(OUT,index=False)

print("Saved:",OUT)
print("Shape:",df.shape)


# Business Notes

## Why these features?

- **Trend** captures long-term business growth.
- **Rolling averages** smooth short-term fluctuations.
- **Lag variables** capture delayed behaviour.
- **Fourier terms** model recurring seasonal patterns without creating many dummy variables.
- These temporal features reduce the risk of attributing organic demand to advertising.

## Feature Pipeline

1. Raw Media
2. Adstock
3. Hill Saturation
4. Calendar Features
5. Trend
6. Rolling Statistics
7. Lag Features
8. Fourier Seasonality
9. Final Feature Matrix

## Interview Questions

1. Why use Fourier terms instead of month dummies?
2. What business insight do lag features provide?
3. Why are rolling averages useful?
4. Why separate trend from seasonality?
5. What happens if trend is omitted from MMM?

## Next Notebook

**10_Feature_Engineering.ipynb**

We'll combine media transformations, interaction features, ratios, elasticities, and feature selection into a production-ready feature matrix for model training.
